In [1]:
import numpy as np

print("NumPy импортирован успешно")

NumPy импортирован успешно


In [6]:
hidden_states = ['Солнечно', 'Дождливо']
observations_names = ['Гулял', 'Магазины', 'Уборка']

A = np.array([
    [0.8, 0.2],
    [0.4, 0.6],
])

B = np.array([
    [0.6, 0.3, 0.1],  # Солнечно: P(Гулял), P(Магазины), P(Уборка)
    [0.1, 0.4, 0.5],  # Дождливо
])

pi0 = np.array([2/3, 1/3])

np.random.seed(42)

n_steps = 10

hidden = np.random.choice([0, 1], p=pi0)
hidden_seq = [hidden]
obs_seq = [np.random.choice([0, 1, 2], p=B[hidden])]

for t in range(n_steps - 1):
    hidden = np.random.choice([0, 1], p=A[hidden])  # строка A для текущего скрытого состояния
    hidden_seq.append(hidden)
    obs_seq.append(np.random.choice([0, 1, 2], p=B[hidden]))  # строка B для текущего скрытого состояния

print("Истинные скрытые состояния:", [hidden_states[s] for s in hidden_seq])
print("Наблюдаемые активности:  ", [observations_names[o] for o in obs_seq])

Истинные скрытые состояния: ['Солнечно', 'Солнечно', 'Солнечно', 'Солнечно', 'Солнечно', 'Солнечно', 'Дождливо', 'Солнечно', 'Солнечно', 'Солнечно']
Наблюдаемые активности:   ['Уборка', 'Гулял', 'Гулял', 'Магазины', 'Магазины', 'Уборка', 'Магазины', 'Гулял', 'Гулял', 'Гулял']


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Почему для генерации наблюдения на шаге t мы используем СКРЫТОЕ состояние именно этого шага t (а не предыдущего или следующего)? Свяжите ответ с ключевым предположением HMM из лекции.</td><td>Ключевое предположение HMM — условная независимость наблюдения от всей истории при условии текущего скрытого состояния: P(o_t | s_1..s_T, o_1..o_{t-1}) = P(o_t | s_t). Иными словами, наблюдение в момент t порождается только скрытым состоянием в этот же момент t (emission-модель B[s_t, o_t]), а не прошлым или будущим состоянием. Поэтому при симуляции сначала генерируется s_t (марковский переход из s_{t-1}), а затем o_t по строке B[s_t]. Это и отражает графическую структуру HMM: наблюдение «прикреплено» к своему скрытому состоянию.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [7]:
def forward(obs_seq, A, B, pi0):
    T = len(obs_seq)
    N = A.shape[0]  # число скрытых состояний
    alpha = np.zeros((T, N))

    # Инициализация (шаг t=0)
    alpha[0] = pi0 * B[:, obs_seq[0]]

    # Рекурсия (шаги t=1, ..., T-1)
    for t in range(1, T):
        for j in range(N):
            alpha[t, j] = np.sum(alpha[t-1] * A[:, j]) * B[j, obs_seq[t]]  # столбец j матрицы A

    return alpha

alpha = forward(obs_seq, A, B, pi0)

p_observations = np.sum(alpha[-1])  # сумма по последней строке alpha

print("Матрица alpha:")
print(alpha)

print(f"\nP(наблюдения | модель) = {p_observations:.8f}")

Матрица alpha:
[[6.66666667e-02 1.66666667e-01]
 [7.20000000e-02 1.13333333e-02]
 [3.72800000e-02 2.12000000e-03]
 [9.20160000e-03 3.49120000e-03]
 [2.62732800e-03 1.57401600e-03]
 [2.73146880e-04 7.34937600e-04]
 [1.53747763e-04 1.98236774e-04]
 [1.21375752e-04 1.49691617e-05]
 [6.18529599e-05 3.32566475e-06]
 [3.04875803e-05 1.43659908e-06]]

P(наблюдения | модель) = 0.00003192


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Почему значение P(наблюдения) получается таким маленьким числом (порядка 10⁻⁴ или меньше)? Это ошибка в коде или ожидаемый результат? (Подсказка: вспомните, сколько существует ВСЕВОЗМОЖНЫХ последовательностей наблюдений длины 10 из 3 возможных активностей.)</td><td>Это ожидаемый результат, а не ошибка. Всевозможных последовательностей наблюдений длины T=10 из 3 символов ровно 3¹⁰ = 59 049. Даже если все они были бы равновероятны, вероятность каждой была бы ≈ 1/59049 ≈ 1.7·10⁻⁵. На практике распределение неравномерно, но «типичная» конкретная последовательность всё равно имеет вероятность порядка 10⁻⁴–10⁻⁵. Формула forward корректно суммирует вероятности по всем 2¹⁰ = 1024 скрытым путям, давая именно эту малую величину. Понимание этого важно для следующих практик: при больших T возникают проблемы численной устойчивости, и переходят к логарифмам (log-space forward).</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [8]:
def viterbi(obs_seq, A, B, pi0):
    T = len(obs_seq)
    N = A.shape[0]

    delta = np.zeros((T, N))  # delta[t,j] = вероятность лучшего пути, оканчивающегося в j на шаге t
    psi = np.zeros((T, N), dtype=int)  # psi[t,j] = какое состояние на шаге t-1 дало этот лучший путь

    delta[0] = pi0 * B[:, obs_seq[0]]

    for t in range(1, T):
        for j in range(N):
            seq_probs = delta[t-1] * A[:, j]
            psi[t, j] = np.argmax(seq_probs)  # индекс состояния, давшего максимум
            delta[t, j] = np.max(seq_probs) * B[j, obs_seq[t]]

    # Восстановление наилучшего пути (идём с конца)
    path = np.zeros(T, dtype=int)
    path[-1] = np.argmax(delta[-1])

    for t in range(T - 2, -1, -1):
        path[t] = psi[t + 1, path[t + 1]]  # индекс момента времени нужен t+1

    return path

decoded_path = viterbi(obs_seq, A, B, pi0)

print("Наблюдения:                 ", [observations_names[o] for o in obs_seq])
print("Истинные скрытые сост.:     ", [hidden_states[s] for s in hidden_seq])
print("Декодированные (Витерби):   ", [hidden_states[s] for s in decoded_path])

accuracy = np.mean(np.array(decoded_path) == np.array(hidden_seq))
print(f"\nТочность декодирования: {accuracy:.2%}")

Наблюдения:                  ['Уборка', 'Гулял', 'Гулял', 'Магазины', 'Магазины', 'Уборка', 'Магазины', 'Гулял', 'Гулял', 'Гулял']
Истинные скрытые сост.:      ['Солнечно', 'Солнечно', 'Солнечно', 'Солнечно', 'Солнечно', 'Солнечно', 'Дождливо', 'Солнечно', 'Солнечно', 'Солнечно']
Декодированные (Витерби):    ['Дождливо', 'Солнечно', 'Солнечно', 'Солнечно', 'Солнечно', 'Солнечно', 'Солнечно', 'Солнечно', 'Солнечно', 'Солнечно']

Точность декодирования: 80.00%


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Насколько высокой оказалась точность декодирования? Все ли дни декодированы правильно? Если есть ошибки — на ваш взгляд, с чем это может быть связано (учитывая, что даже в дождливую погоду друг иногда гуляет, и наоборот)?</td><td>Точность высокая (обычно 80–100% на 10 шагах при seed=42), но отдельные дни могут быть декодированы неверно. Причина — вероятностная природа эмиссий: наблюдение «Гулял» с вероятностью 0.1 может возникнуть и в дождливый день, а «Уборка» — с вероятностью 0.1 в солнечный. Витерби находит наиболее вероятный **совместный** путь, а не покомпонентно наиболее вероятное состояние в каждый момент (это был бы forward-backward / posterior decoding). Если наблюдение нетипично для истинного состояния, а типично для другого, алгоритм может «переключить» состояние — особенно на коротких сегментах, где переход A[i,j] дёшев. Кроме того, ошибка в одном дне может «тянуть» за собой соседние из-за глобальной оптимизации пути. С ростом T и при более контрастных B точность растёт.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [9]:
hidden_arr = np.array(hidden_seq)
obs_arr = np.array(obs_seq)

# Оценка матрицы переходов A по частоте переходов
A_est = np.zeros((2, 2))
for i in range(2):
    for j in range(2):
        mask_i = hidden_arr[:-1] == i
        A_est[i, j] = np.sum((hidden_arr[:-1] == i) & (hidden_arr[1:] == j)) / np.sum(mask_i)

print("Оценённая матрица A:")
print(A_est)
print("Истинная матрица A:")
print(A)

# Оценка матрицы эмиссии B по частоте наблюдений при каждом скрытом состоянии
B_est = np.zeros((2, 3))
for i in range(2):
    for k in range(3):
        B_est[i, k] = np.sum((hidden_arr == i) & (obs_arr == k)) / np.sum(hidden_arr == i)

print("\nОценённая матрица B:")
print(B_est)
print("Истинная матрица B:")
print(B)

Оценённая матрица A:
[[0.875 0.125]
 [1.    0.   ]]
Истинная матрица A:
[[0.8 0.2]
 [0.4 0.6]]

Оценённая матрица B:
[[0.55555556 0.22222222 0.22222222]
 [0.         1.         0.        ]]
Истинная матрица B:
[[0.6 0.3 0.1]
 [0.1 0.4 0.5]]


Шаблон для таблиц
<table>
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Насколько близки оценённые матрицы A_est и B_est к истинным A и B? При n_steps=10 (задание 1) оценки могут получиться не очень точными — почему? Что бы вы предложили изменить, чтобы получить более точные оценки? (Подсказка: вспомните состоятельность MLE-оценок из лекции 4.)</td><td>При T=10 оценки заметно отличаются от истинных: например, в A_est некоторые строки могут даже содержать 0 или 1 вместо 0.8/0.2. Причина — крайне малое число «испытаний» на каждую строку: для A есть всего 9 переходов, и на состояние «Дождливо» их приходится совсем немного; для B — 10 наблюдений, распределённых по двум состояниям. MLE-оценки состоятельны: p̂ → p по вероятности при n → ∞ (закон больших чисел), но при малых n у них большая дисперсия, а при нулевом числе наблюдений в ячейке оценка вообще не определена (0/0). Чтобы получить точные оценки, нужно увеличить длину последовательности T (сотни–тысячи шагов) — тогда частоты сойдутся к истинным вероятностям. На практике, когда скрытые состояния не наблюдаются, вместо прямого подсчёта используют EM-алгоритм Баума–Велша, который итеративно уточняет и A, и B по постериорным вероятностям состояний.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>